## Nullable Fields

Wie lässt sich erlauben, dass ein Field den Wert `None` annimmt, etwa wenn eine API ein Feld mit `null` liefert? Ein Field ist standardmäßig nicht nullable: Der Typ in der Annotation legt fest, was gültig ist, und `None` gehört bei `int` nicht dazu. Mit `| None` in der Annotation wird `None` zusätzlich erlaubt.

**Danach ist klar:**
- warum `None` bei einem `int`-Field abgelehnt wird,
- wie `int | None` das Field nullable macht und `= None` es zusätzlich optional,
- dass `Union[int, None]`, `Optional[int]` und `int | None` dasselbe bedeuten, `int | None` aber die übliche Schreibweise ist.

Voraussetzung: Notebook `06 - Required vs Optional Fields`.

In [ ]:
# `BaseModel` definiert das Model, `ValidationError` fängt später die Fehlerfälle ab.
from pydantic import BaseModel, ValidationError

In [ ]:
# Das Field ist ein reiner `int` und ohne Default Pflicht.
class Model(BaseModel):
    field: int

**Vorhersage:** Das Field ist ein reiner `int`. Was meldet Pydantic bei `Model(field=None)` und bei `Model()`? Sind die beiden Fehler gleich?

<details>
<summary>Erklärung anzeigen</summary>

**Was der Code macht:** Beide Aufrufe scheitern, aber aus verschiedenen Gründen. `field=None` ist zwar angegeben, aber kein `int` (`int_type`). Bei `Model()` fehlt das Pflicht-Field ganz (`missing`).

Kurz gesagt: „`None` angegeben“ und „nichts angegeben“ sind zwei verschiedene Fälle.
</details>

In [ ]:
try:
    Model(field=None)
except ValidationError as exc:
    print(exc)

In [ ]:
try:
    Model()
except ValidationError as exc:
    print(exc)

### Nullable Fields

Wie wird `None` als Wert erlaubt? Die Annotation wird um `| None` erweitert. Das Field ist dann nullable. Ob es auch fehlen darf, ist damit noch nicht entschieden.

In [ ]:
# `int | None` erlaubt `int` und `None`. Ohne Default bleibt das Field Pflicht, `Model()` würde `missing` melden.
class Model(BaseModel):
    field: int | None

In [ ]:
# `None` ist jetzt ein gültiger Wert.
Model(field=None)

In [ ]:
# Ein `int` bleibt gültig.
Model(field=1)

Wie darf das Field zusätzlich beim Erzeugen fehlen? Dafür sorgt ein Default, wie schon in `06 - Required vs Optional Fields`.

In [ ]:
# Der Default `None` macht das Field zusätzlich optional.
class Model(BaseModel):
    field: int | None = None

In [ ]:
# Ohne Angabe greift der Default.
Model()

In [ ]:
# `None` ausdrücklich zu übergeben ist ebenfalls gültig.
Model(field=None)

In [ ]:
# Ein `int` bleibt gültig.
Model(field=10)

### Schreibweisen: `Union`, `Optional` und `|`

Für dieselbe Annotation gibt es mehrere Schreibweisen. Macht das für Pydantic einen Unterschied?

In [ ]:
# `Union[int, None]` ist die ältere Schreibweise für `int | None`.
from typing import Union

class Model(BaseModel):
    field: Union[int, None] = None

In [ ]:
# `Optional` kommt wie `Union` aus `typing`.
from typing import Optional

In [ ]:
# `Optional[int]` ist eine Kurzform für `Union[int, None]`.
class Model(BaseModel):
    field: Optional[int] = None

**Vorhersage:** Wie sieht `model_fields` für `Optional[int] = None` aus? Ist das Field Pflicht, und unterscheidet es sich von `int | None = None`?

<details>
<summary>Erklärung anzeigen</summary>

**Was der Code macht:** `model_fields` zeigt `required=False` und den Default `None`. Die Annotation ist dieselbe wie bei `int | None`, Pydantic behandelt alle drei Schreibweisen gleich.

Kurz gesagt: `int | None` ist die kanonische Form, `Union` und `Optional` sind gleichwertig, aber älter.
</details>

In [ ]:
Model.model_fields

**Vorhersage:** `Optional[int]` steht jetzt ohne Default. Ist das Field Pflicht oder optional?

<details>
<summary>Erklärung anzeigen</summary>

**Was der Code macht:** `model_fields` zeigt `required=True`. `Optional` heißt nur, dass `None` ein gültiger Wert ist, nicht dass das Field fehlen darf.

Kurz gesagt: Nullable (`None` erlaubt) und optional (darf fehlen) sind zwei getrennte Eigenschaften.
</details>

In [ ]:
class Model(BaseModel):
    field: Optional[int]

In [ ]:
Model.model_fields

In [ ]:
# Kanonische Schreibweise: `int | None` mit Default `None`.
class Model(BaseModel):
    field: int | None = None

In [ ]:
# Wie zuvor entsteht die Instanz mit dem Default `None`.
Model()

### Zusammenfassung

- Ein Field wie `field: int` lehnt `None` ab, und ohne Default fehlt es als Pflicht-Field (`missing`).
- `int | None` macht das Field nullable, `= None` zusätzlich optional.
- Nullable und optional sind getrennte Eigenschaften: `Optional[int]` ohne Default bleibt Pflicht.
- `Union[int, None]`, `Optional[int]` und `int | None` sind gleichwertig, `int | None` ist die kanonische Schreibweise.

Weiter mit `08 - Combining Nullable and Optional`.